# Banking RAG Evaluation

This notebook loads the two policy PDFs, validates the 16 expected section IDs, builds the FAISS knowledge base, runs retrieval, creates answers, evaluates the outputs, and saves the scorecard and per-question results.


In [ ]:
import json
import platform
import time
from pathlib import Path

import streamlit

from src.chunker import chunk_policy_sections
from src.config import CHUNK_OVERLAP, CHUNK_SIZE, DATA_DIR, EMBEDDING_MODEL, EVALUATION_DIR, PROJECT_ROOT, TOP_K
from src.embeddings import embed_texts, load_embedding_model
from src.evaluation import (
    citation_verification_score,
    compute_scorecard,
    correctness_score,
    faithfulness_score,
    load_questions,
    retrieval_recall_required_sections,
    response_relevancy_score,
    safe_handling_score,
    save_scorecard,
    write_results_csv,
)
from src.generator import generate_answer
from src.pdf_loader import extract_pdf_sections, validate_loaded_pdfs
from src.retriever import retrieve_top_chunks
from src.vector_store import build_faiss_index

print('Banking RAG evaluation')
print('Python', platform.python_version())
print('Streamlit', streamlit.__version__)


In [ ]:
pdf_paths = sorted(DATA_DIR.glob('*.pdf'))
print('pdf_paths', [p.name for p in pdf_paths])
validation_report, missing_sections = validate_loaded_pdfs(pdf_paths)
print('missing_sections', missing_sections)
sections = []
for pdf_path in pdf_paths:
    page_count, page_sections, _ = extract_pdf_sections(pdf_path)
    sections.extend(page_sections)
print(pdf_path.name, 'page_count', page_count, 'sections', len(page_sections))
chunks = chunk_policy_sections(sections)
print('chunk_count', len(chunks))
embedding_model = load_embedding_model(EMBEDDING_MODEL)
texts = [chunk['text'] for chunk in chunks]
embeddings = embed_texts(embedding_model, texts)
faiss_index = build_faiss_index(embeddings)
print('embedding_dimension', embeddings.shape[1])
print('faiss_index', faiss_index)


In [ ]:
questions = load_questions()
results = []
for entry in questions:
    start = time.perf_counter()
    retrieved = retrieve_top_chunks(chunks, entry['question'], faiss_index, embedding_model, top_k=TOP_K)
    output = generate_answer(entry['question'], retrieved)
    elapsed = time.perf_counter() - start
    retrieved_sections = [item['section_id'] for item in retrieved]
    supporting_text = ' '.join(item['text'] for item in retrieved)
    retrieval_recall = retrieval_recall_required_sections(retrieved_sections, entry.get('required_sections', []))
    faithfulness = faithfulness_score(output['answer'], [item['text'] for item in retrieved])
    correctness = correctness_score(entry['reference_answer'], output['answer']) if entry['answerable'] else 1.0
    relevancy = response_relevancy_score(entry['question'], output['answer'])
    safe_score = safe_handling_score(output['answer'], entry['question'])
    supported_links, correct_claims = citation_verification_score(output['answer'], output.get('citations', []), supporting_text)
    results.append({
        'id': entry['id'],
        'split': entry['split'],
        'question': entry['question'],
        'answerable': entry['answerable'],
        'reference_answer': entry['reference_answer'],
        'actual_answer': output['answer'],
        'retrieved_passages': retrieved,
        'retrieved_section_ids': retrieved_sections,
        'similarity_scores': [item['score'] for item in retrieved],
        'emitted_citations': output.get('citations', []),
        'supported_citation_links': supported_links,
        'correct_factual_claims': correct_claims,
        'retrieval_evidence_recall': retrieval_recall,
        'faithfulness': faithfulness,
        'correctness': correctness,
        'response_relevancy': relevancy,
        'safe_handling': safe_score,
        'latency_seconds': elapsed,
        'required_sections': entry.get('required_sections', []),
    })

print('evaluated', len(results), 'questions')
print('dev', sum(1 for r in results if r['split']=='development'))
print('final', sum(1 for r in results if r['split']=='final_test'))
print('challenge', sum(1 for r in results if r['split']=='challenge'))

scorecard = compute_scorecard(results)
save_scorecard(scorecard, EVALUATION_DIR / 'scorecard.json')
with open(EVALUATION_DIR / 'results.json', 'w', encoding='utf-8') as fh:
    json.dump({'results': results, 'scorecard': scorecard}, fh, indent=2)
import pandas as pd
write_results_csv(results, EVALUATION_DIR / 'question_level_results.csv')
print('saved evaluation artifacts')


In [ ]:
config = {
    'python_version': platform.python_version(),
    'streamlit_version': streamlit.__version__,
    'pymupdf_version': '1.24.10',
    'sentence_transformers_version': '6.1.0',
    'faiss_version': '1.15.1',
    'embedding_model': EMBEDDING_MODEL,
    'embedding_dimension': embeddings.shape[1],
    'llm_provider': 'openai',
    'llm_model': 'gpt-4o-mini',
    'chunk_size': CHUNK_SIZE,
    'chunk_overlap': CHUNK_OVERLAP,
    'top_k': TOP_K,
    'index_type': 'IndexFlatIP',
}
with open(EVALUATION_DIR / 'config.json', 'w', encoding='utf-8') as fh:
    json.dump(config, fh, indent=2)
print(config)
